# Gridsecure - Exploratory Data Analysis (EDA)
**Project Title**: Gridsecure (Electricity Theft Detection System)  
**Course**: Data Analytics & AI (JIIT Summer Internship Program)  
**Role**: Member 2 - Exploratory Data Analysis  

---
### Objective
Perform thorough exploratory data analysis on the SGCC electricity consumption dataset (`electricity_theft_dataset_with_clusters_V2.csv`) to understand feature distributions, identify missing data, detect outliers, analyze correlations, and discover key behavioral patterns associated with electricity theft.


## 1. Environment Setup & Library Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.feature_selection import mutual_info_classif

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries imported successfully!")


## 2. Load Dataset


In [ ]:
# Load preprocessed dataset from Member 1
# In Google Colab, upload 'electricity_theft_dataset_with_clusters_V2.csv' to the files sidebar
df = pd.read_csv('electricity_theft_dataset_with_clusters_V2.csv')

print("Dataset Shape:", df.shape)
df.head()


## 3. Data Cleaning & Overview


In [ ]:
# Drop location metadata if present
location_cols = ["State", "City", "Locality"]
df.drop(columns=[col for col in location_cols if col in df.columns], inplace=True)

print("Columns in dataset:", df.columns.tolist()[:15])
print("\nDataset Info:")
df.info()

print("\nMissing Values Total:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())


## 4. Class Distribution & Target Analysis


In [ ]:
# Check target class distribution (Theft_Flag)
print("Theft Flag Counts:")
print(df["Theft_Flag"].value_counts())

print("\nPercentage Distribution:")
print(df["Theft_Flag"].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(x="Theft_Flag", data=df, hue="Theft_Flag", legend=False, palette="Set2")
plt.title("Distribution of Normal (0) vs Electricity Theft (1) Cases")
plt.xlabel("Theft Flag (0 = Normal, 1 = Theft)")
plt.ylabel("Consumer Count")
plt.show()


## 5. Consumer Type Analysis


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="Consumer_Type", data=df, hue="Consumer_Type", legend=False, palette="viridis")
plt.title("Consumer Type Distribution")
plt.xticks(rotation=45)
plt.show()

# Cross-tabulation of Consumer Type vs Theft Flag
ct = pd.crosstab(df["Consumer_Type"], df["Theft_Flag"])
print("Cross-tabulation:\n", ct)

plt.figure(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')
plt.title("Theft Cases Across Consumer Types")
plt.show()


## 6. Numerical Feature Distributions & Outliers


In [ ]:
important_features = [
    "Avg_Consumption",
    "Median_Consumption",
    "Max_Consumption",
    "Std_Consumption",
    "Consumption_Range",
    "Behavioural_Anomaly_Score"
]

# Skewness calculation
num_cols = df.select_dtypes(include=np.number).columns
skewness = df[num_cols].skew().sort_values(ascending=False)
print("Top 10 Positively Skewed Features:\n", skewness.head(10))

# Boxplots for outlier detection
for col in important_features:
    if col in df.columns:
        plt.figure(figsize=(8, 3))
        sns.boxplot(x=df[col], color="skyblue")
        plt.title(f"Boxplot of {col}")
        plt.show()
        
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"Number of IQR Outliers in {col}: {len(outliers)}")


## 7. Correlation Analysis & Heatmap


In [ ]:
# Correlation of numeric features with Theft_Flag
corr_target = df.corr(numeric_only=True)["Theft_Flag"].sort_values(ascending=False)
print("Top Correlated Features with Theft_Flag:\n", corr_target.head(15))

top20 = corr_target.abs().sort_values(ascending=False).head(20).index

plt.figure(figsize=(12, 8))
sns.heatmap(df[top20].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Top 20 Features Correlation Heatmap")
plt.show()


## 8. Bivariate & Behavioral Visualizations


In [ ]:
# Violin plot of Avg_Consumption vs Theft_Flag
plt.figure(figsize=(7, 4))
sns.violinplot(x="Theft_Flag", y="Avg_Consumption", data=df, hue="Theft_Flag", legend=False, palette="muted")
plt.title("Avg Consumption Distribution by Theft Flag")
plt.show()

# Scatter plot: Avg Consumption vs Max Consumption
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df.sample(5000, random_state=42), x="Avg_Consumption", y="Max_Consumption", hue="Theft_Flag", alpha=0.7)
plt.title("Avg vs Max Consumption (Sampled 5k)")
plt.show()

# Seasonal Consumption Comparison
season_cols = ["Summer_Avg", "Monsoon_Avg", "Winter_Avg"]
if all(col in df.columns for col in season_cols):
    df[season_cols].mean().plot(kind="bar", color=["orange", "teal", "dodgerblue"])
    plt.title("Average Consumption by Season")
    plt.ylabel("kWh")
    plt.show()


## 9. Cluster Analysis & Mutual Information


In [ ]:
# Behaviour Cluster vs Theft Flag
if "Behaviour_Cluster" in df.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(x="Behaviour_Cluster", hue="Theft_Flag", data=df, palette="Set1")
    plt.title("Theft Flag Distribution across Behavior Clusters")
    plt.show()

# Mutual Information Feature Importance (sampled for speed)
sample_df = df.sample(min(5000, len(df)), random_state=42)
X_mi = sample_df.drop(columns=["Theft_Flag", "CONS_NO"], errors="ignore").select_dtypes(include=np.number)
X_mi = X_mi.replace([np.inf, -np.inf], np.nan).fillna(X_mi.median())
y_mi = sample_df["Theft_Flag"]

mi = mutual_info_classif(X_mi, y_mi)
mi_series = pd.Series(mi, index=X_mi.columns).sort_values(ascending=False)

print("Top 15 Features by Mutual Information:\n", mi_series.head(15))
plt.figure(figsize=(10, 5))
mi_series.head(15).plot(kind="barh", color="purple")
plt.gca().invert_yaxis()
plt.title("Top 15 Features by Mutual Information Score")
plt.xlabel("Mutual Information")
plt.show()


## 10. Summary & Export Report


In [ ]:
# Save summary statistics to CSV
summary = df.describe(include='all').T
summary.to_csv("EDA_Report.csv")
print("EDA Report saved to 'EDA_Report.csv' successfully!")
